# Google Search Hits vs. Citation Count

Reproduces the Google search hits analysis (Section 3.1 and Appendix "More Results on Citation As a Proxy"):

1. Correlation between citation count and Google hit count (raw and log2 scale).
2. Per-citation-bucket statistics on all crawled papers (N = 5,179 in the paper).
3. Search-query string length per bucket (explains the outliers in low-citation buckets).
4. Outlier filtering: remove hit counts above the 80th percentile within each bucket (N = 4,185).
   Other cutoffs (75 / 90 / 95%) are computed as a sensitivity check.

**Input:** `results/google_search_hits/combined_google_results.csv` (from `crawl_google_hits.py`).
**Output:** statistics tables and a scatter plot in `results/google_search_hits/`.

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import LogLocator
from scipy.stats import pearsonr, spearmanr
from tabulate import tabulate

sys.path.insert(0, str(Path.cwd().parent))  # run this notebook from its own folder
import config

OUT_DIR = config.GOOGLE_HITS_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(OUT_DIR / "combined_google_results.csv")
print(f"{len(df)} papers with a valid hit count")
df.describe()

## 1. Correlation on the raw scale

In [ ]:
pearson_corr, _ = pearsonr(df["citation"], df["number of hits"])
spearman_corr, _ = spearmanr(df["citation"], df["number of hits"])
print(f"Pearson:  {pearson_corr:.4f}")
print(f"Spearman: {spearman_corr:.4f}")

## 2. Log2-scale scatter plot and correlation

Zero counts are replaced by 0.1 so that they can be shown on a log scale. The bucket
statistics below are computed on these values as well (this matches the tables in the paper).

In [ ]:
df["number of hits"] = df["number of hits"].replace(0, 0.1)
df["citation"] = df["citation"].replace(0, 0.1)

log_x = np.log2(df["citation"])
log_y = np.log2(df["number of hits"])
slope, intercept = np.polyfit(log_x, log_y, 1)
log_x_fit = np.linspace(log_x.min(), log_x.max(), 500)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df["citation"], df["number of hits"], alpha=0.3, color="#1f77b4")
ax.plot(2 ** log_x_fit, 2 ** (slope * log_x_fit + intercept), color="red", linewidth=2,
        label="Linear fit (log2 space)")
ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.xaxis.set_major_locator(LogLocator(base=2, numticks=10))
ax.yaxis.set_major_locator(LogLocator(base=2, numticks=10))
ax.set_xlabel("Citation (Log2 Scale)", fontsize=16)
ax.set_ylabel("Google Hits (Log2 Scale)", fontsize=16)
ax.set_title("Citation vs Google Hits (Log2 Scale)", fontsize=18)
ax.grid(True, which="both", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "citation_vs_google_hits.pdf", dpi=300, bbox_inches="tight")
plt.show()

pearson_corr, _ = pearsonr(log_x, log_y)
spearman_corr, _ = spearmanr(log_x, log_y)
print(f"Pearson (log2):  {pearson_corr:.4f}")
print(f"Spearman (log2): {spearman_corr:.4f}")

## 3. Statistics per citation bucket (all crawled papers)

In [ ]:
df["citation_bucket"] = pd.cut(df["citation"], bins=config.CITATION_BIN_EDGES,
                               labels=config.CITATION_BIN_LABELS, right=True)


def compute_strlen(question: str) -> int:
    """Total length of the quoted parts (title + author names) of a search query."""
    return sum(len(part) for part in re.findall(r'"(.*?)"', question))


df["strlen_title_authors"] = df["question"].apply(compute_strlen)


def bucket_stats(data: pd.DataFrame) -> pd.DataFrame:
    """Hit-count and query-length statistics per citation bucket, including the coefficient of variation."""
    table = data.groupby("citation_bucket", observed=False).agg(
        count=("number of hits", "count"),
        mean_hits=("number of hits", "mean"),
        std_hits=("number of hits", "std"),
        min_hits=("number of hits", "min"),
        q25_hits=("number of hits", lambda x: x.quantile(0.25)),
        q50_hits=("number of hits", "median"),
        q75_hits=("number of hits", lambda x: x.quantile(0.75)),
        max_hits=("number of hits", "max"),
        avg_strlen_title_authors=("strlen_title_authors", "mean"),
        std_strlen_title_authors=("strlen_title_authors", "std"),
    ).reset_index()
    table["cv_hits"] = table["std_hits"] / table["mean_hits"]
    return table


table_full = bucket_stats(df)
table_full.to_csv(OUT_DIR / "citation_bucket_stats_full.csv", index=False)
print(tabulate(table_full, headers="keys", tablefmt="github", showindex=False, floatfmt=".2f"))

## 4. Short queries cause spurious hits

Papers in the 0-2 bucket sorted by query length: very short titles (e.g. "Is this tragic or comic")
match unrelated web pages and receive huge hit counts.

In [ ]:
low_bucket = df[df["citation_bucket"] == "0-2"].sort_values("strlen_title_authors")
low_bucket[["question", "citation", "number of hits", "strlen_title_authors"]].head(10)

## 5. Outlier filtering per bucket

Hit counts above the given percentile within each citation bucket are removed. The paper uses the
80th percentile (cutoff = 0.2), which keeps 4,185 papers.

In [ ]:
def remove_top_percentile(group: pd.DataFrame, cutoff: float) -> pd.DataFrame:
    """Drop rows whose hit count is above the (1 - cutoff) quantile of the group."""
    threshold = group["number of hits"].quantile(1 - cutoff)
    return group[group["number of hits"] <= threshold]


filtered_tables = {}
for keep_pct in [75, 80, 90, 95]:
    cutoff = 1 - keep_pct / 100
    df_filtered = pd.concat(
        [remove_top_percentile(group, cutoff) for _, group in df.groupby("citation_bucket", observed=False)]
    )
    filtered_tables[keep_pct] = bucket_stats(df_filtered)
    filtered_tables[keep_pct].to_csv(OUT_DIR / f"citation_bucket_stats_{keep_pct}.csv", index=False)
    print(f"Keep <= {keep_pct}th percentile: {len(df_filtered)} papers")

print("\nMain result (80th percentile):")
print(tabulate(filtered_tables[80][["citation_bucket", "count", "mean_hits", "std_hits", "min_hits", "q25_hits",
                                    "q50_hits", "q75_hits", "max_hits", "cv_hits"]],
               headers="keys", tablefmt="github", showindex=False, floatfmt=".2f"))